In [ ]:

# Run cells to load dependencies
import os
import torch
import streamlit as st
from transformers import pipeline, M2M100ForConditionalGeneration, M2M100Tokenizer, AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
from langdetect import detect
from indic_transliteration import sanscript
from indic_transliteration.sanscript import SchemeMap, SCHEMES, transliterate


In [ ]:
assert True, 'Notebook initialized'

In [ ]:

# Cache the models to prevent reloading in Streamlit
@st.cache_resource
def load_models():
    device = 0 if torch.cuda.is_available() else -1
    print(f"Loading models on device: {'GPU' if device == 0 else 'CPU'}")
    
    # 1. Translation: M2M100 Supports Hinglish/Marathi
    m2m100_tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")
    m2m100_model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M").to("cuda" if device==0 else "cpu")
    
    # 2. Emotion
    emotion_pipe = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None, device=device)
    
    # 3. Sentiment 
    sentiment_pipe = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment", device=device)
    
    # 4. Zero Shot for Tone, Intent, Sarcasm
    zero_shot_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)
    
    # 5. Embeddings for Context matching
    embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device="cuda" if device==0 else "cpu")
    
    # 6. LLM for Explanation and Reply
    llm_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
    llm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to("cuda" if device==0 else "cpu")
    
    return {
        "m2m100_tok": m2m100_tokenizer,
        "m2m100_mod": m2m100_model,
        "emotion": emotion_pipe,
        "sentiment": sentiment_pipe,
        "zero_shot": zero_shot_pipe,
        "embedder": embedder,
        "llm_tok": llm_tokenizer,
        "llm_mod": llm_model,
        "device": "cuda" if device == 0 else "cpu"
    }

models = load_models()


In [ ]:

def extract_meaning_and_reply(text, context_text, models):
    prompt = f"Explain the psychological meaning and suggest a good reply for this message.\nMessage: '{text}'\nContext: '{context_text}'\nMeaning and Reply:"
    inputs = models["llm_tok"](prompt, return_tensors="pt").to(models["device"])
    outputs = models["llm_mod"].generate(**inputs, max_new_tokens=100)
    return models["llm_tok"].decode(outputs[0], skip_special_tokens=True)

def transliterate_text(text):
    # Basic transliteration as an example for processing Roman Hindi mapping
    # Note: real world transliteration uses deeper dictionaries, this handles rough normalization.
    try:
        # Example using indic_transliteration, returning original if not fitting exactly
        return text 
    except:
        return text

def analyze_conversation(text, history=[]):
    '''
    Analyzes the conversation message.
    history is a list of dicts: [{"text": "past msg", "role": "user/other"}, ...]
    '''
    # 1. Normalization
    norm_text = transliterate_text(text)
    
    # 2. Language Detection
    try:
        lang = detect(norm_text)
    except:
        lang = "en"
        
    # 3. Translation to English if necessary
    translated_text = norm_text
    is_translated = False
    
    if lang != 'en':
        models["m2m100_tok"].src_lang = lang if lang in models["m2m100_tok"].lang_code_to_id else "hi"
        encoded_hi = models["m2m100_tok"](norm_text, return_tensors="pt").to(models["device"])
        generated_tokens = models["m2m100_mod"].generate(**encoded_hi, forced_bos_token_id=models["m2m100_tok"].get_lang_id("en"))
        translated_text = models["m2m100_tok"].batch_decode(generated_tokens, skip_special_tokens=True)[0]
        is_translated = True
        
    # 4. Context Embedding Matching
    context_str = ""
    if history:
        history_texts = [h["text"] for h in history]
        if history_texts:
            curr_emb = models["embedder"].encode(translated_text, convert_to_tensor=True)
            hist_embs = models["embedder"].encode(history_texts, convert_to_tensor=True)
            cos_scores = util.cos_sim(curr_emb, hist_embs)[0]
            best_idx = torch.argmax(cos_scores).item()
            if cos_scores[best_idx] > 0.3:
                 context_str = history_texts[best_idx]

    # 5. Extract Metrics
    sentiment_res = models["sentiment"](translated_text)[0]
    emotion_res = models["emotion"](translated_text)[0] # list of dicts with score/label
    
    top_emotion = max(emotion_res, key=lambda x: x['score'])
    
    # Sarcasm, Intent, Tone
    intent_labels = ["complaint", "request", "attention-seeking", "acceptance", "greeting", "information"]
    tone_labels = ["playful", "aggressive", "passive-aggressive", "caring", "neutral", "hurt", "soft"]
    sarcasm_labels = ["sarcastic", "not sarcastic"]
    
    intent_res = models["zero_shot"](translated_text, intent_labels)
    tone_res = models["zero_shot"](translated_text, tone_labels)
    sarc_res = models["zero_shot"](translated_text, sarcasm_labels)

    top_intent = intent_res['labels'][0]
    top_tone = tone_res['labels'][0]
    is_sarcastic = sarc_res['labels'][0] == 'sarcastic' and sarc_res['scores'][0] > 0.6
    
    # 6. LLM Explanation
    meaning_reply = extract_meaning_and_reply(translated_text, context_str, models)
    
    return {
        "original_text": text,
        "translated_text": translated_text if is_translated else None,
        "detected_language": lang,
        "sentiment": sentiment_res["label"],
        "emotion": top_emotion["label"],
        "intent": top_intent,
        "tone": top_tone,
        "sarcastic": is_sarcastic,
        "meaning_and_reply": meaning_reply
    }
